In [25]:
# Package importation
import os 
%load_ext autoreload
%autoreload 2
os.chdir("..")
import sys
sys.path.append(os.getcwd())
from smc import abc_smc, perm_abc_smc
from over_sampling import perm_abc_smc_os
from under_matching import perm_abc_smc_um  
from models.Gaussian_with_no_summary_stats import GaussianWithNoSummaryStats
from jax import random
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
from distances import optimal_index_distance
from kernels import KernelTruncatedRW

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
key = random.PRNGKey(0)
key, subkey = random.split(key)
K = 10
n = 10
sigma0 = 10
alpha, beta = 5,5
model = GaussianWithNoSummaryStats(K = K, n_obs= n, sigma_0 = sigma0, alpha = alpha, beta = beta)
true_theta = model.prior_generator(subkey, 1)
# true_theta.loc = np.linspace(-2*sigma0, 2*sigma0, K)[None,:, None]
true_theta.glob = np.array([1.])[None,:]
key, subkey = random.split(key)
y_obs = model.data_generator(subkey, true_theta)
print(y_obs.shape)

(1, 10, 10)


In [28]:
n_particles = 1000
alpha_M = .95
kernel = KernelTruncatedRW
n_abc = 100000
key, subkey = random.split(key)
thetas = model.prior_generator(subkey, n_abc)
key, subkey = random.split(key)
zs = model.data_generator(subkey, thetas)
dists_perm, _, _, _ = optimal_index_distance(model, zs, y_obs)
epsilon = np.quantile(dists_perm, .01)
print(epsilon)

1.73603187084198


# permSMC

In [36]:
key, subkey = random.split(key)
alpha_epsilon = .95
test_permsmc = perm_abc_smc(key = subkey, model = model, n_particles= n_particles, y_obs = y_obs, alpha_epsilon= alpha_epsilon, epsilon_target = epsilon, kernel = kernel, update_weights_distance= False)

False
Iteration 0: Epsilon = inf, ESS = 1000 Acc. rate = 100% Numb. unique particles = 1000
Iteration 2: Espilon = 8.5075, ESS = 950 Acc. rate = 23.58% Uniqueness rate particules = 100.0% Uniqueness rate components = 99.9% Global parameters uniqueness rate = 100.0%

Iteration 3: Espilon = 7.7703, ESS = 902 Acc. rate = 25.72% Uniqueness rate particules = 100.0% Uniqueness rate components = 99.9% Global parameters uniqueness rate = 100.0%

Iteration 4: Espilon = 7.2832, ESS = 856 Acc. rate = 23.71% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 5: Espilon = 6.8241, ESS = 813 Acc. rate = 21.28% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 6: Espilon = 6.4997, ESS = 772 Acc. rate = 24.09% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 7: Espilon = 6.1856,

# Over-Sampling

In [30]:
key, subkey = random.split(key)
M0s = np.array([10*K, 5*K, 3*K, 2.5*K, 2*K ,1.5*K, 1.2*K], dtype=int)
alpha_epsilon = .95
thetas = model.prior_generator(subkey, 10000, n_silos= np.max(M0s))
key, subkey = random.split(key)
zs = model.data_generator(subkey, thetas)
print(zs.shape)
epsilons_os = []
for M0 in M0s:
    dists_os, _, _, _ = optimal_index_distance(model, zs[:,:M0], y_obs, M = M0)
    epsilon_os = np.quantile(dists_os, alpha_epsilon)
    epsilons_os.append(epsilon_os)
    print('M0 = {} epsilon = {}'.format(M0, epsilon_os))

(10000, 100, 10)
M0 = 100 epsilon = 0.682169336080551
M0 = 50 epsilon = 0.9780219465494155
M0 = 30 epsilon = 1.4814373850822447
M0 = 25 epsilon = 1.7888628661632526
M0 = 20 epsilon = 2.3544289112091064
M0 = 15 epsilon = 3.5593046069145187
M0 = 12 epsilon = 5.234821677207946


## No Duplicate

In [31]:
key, subkey = random.split(key)
M0 = 25
test_os = perm_abc_smc_os(key = subkey, model = model, n_particles= n_particles, y_obs = y_obs, M_0= M0, alpha_M= alpha_M, epsilon = epsilon, kernel = kernel, verbose = 1, update_weights_distance= False)

Iteration 0: M = 25 Epsilon = 1.73603187084198, ESS = 941 Acc. rate = 100% Numb. unique particles = 1000

Iteration 1: M = 24 Espilon = 1.7360, ESS = 926 Acc. rate = 36.93% Uniqueness rate particules = 100.0% Uniqueness rate components = 99.9% Global parameters uniqueness rate = 100.0%

Iteration 2: M = 23 Espilon = 1.7360, ESS = 898 Acc. rate = 36.41% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 3: M = 22 Espilon = 1.7360, ESS = 868 Acc. rate = 36.98% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 4: M = 21 Espilon = 1.7360, ESS = 826 Acc. rate = 42.25% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 5: M = 20 Espilon = 1.7360, ESS = 780 Acc. rate = 39.74% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters unique

## Duplicate

In [32]:
test_os_duplicate = perm_abc_smc_os(key = subkey, model = model, n_particles= n_particles, y_obs = y_obs, M_0= M0, alpha_M= alpha_M, epsilon = epsilon, kernel = kernel, verbose = 1, duplicate= True, update_weights_distance= False)

Iteration 0: M = 25 Epsilon = 1.73603187084198, ESS = 941 Acc. rate = 100% Numb. unique particles = 1000

Iteration 1: M = 24 Espilon = 1.7360, ESS = 926 Acc. rate = 36.93% Uniqueness rate particules = 100.0% Uniqueness rate components = 99.9% Global parameters uniqueness rate = 100.0%

Iteration 2: M = 23 Espilon = 1.7360, ESS = 898 Acc. rate = 36.41% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 3: M = 22 Espilon = 1.7360, ESS = 868 Acc. rate = 36.98% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 4: M = 21 Espilon = 1.7360, ESS = 826 Acc. rate = 42.25% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters uniqueness rate = 100.0%

Iteration 5: M = 20 Espilon = 1.7360, ESS = 780 Acc. rate = 39.74% Uniqueness rate particules = 100.0% Uniqueness rate components = 100.0% Global parameters unique

# Under-Matching

In [34]:
key, subkey = random.split(key)
thetas = model.prior_generator(subkey, 10000)
key, subkey = random.split(key)
zs = model.data_generator(subkey, thetas)

for L0 in np.arange(1, K+1):
    dists_um, _, _, _ = optimal_index_distance(model, zs, y_obs, L = L0)
    epsilon_um = np.quantile(dists_um, alpha_epsilon)
    print(f"L0 = {L0} epsilon = {epsilon_um}")

L0 = 1 epsilon = 0.2506378337740897
L0 = 2 epsilon = 0.4155756533145904
L0 = 3 epsilon = 0.5815757513046265
L0 = 4 epsilon = 0.7769739300012588
L0 = 5 epsilon = 1.0471002161502836
L0 = 6 epsilon = 1.4771112561225885
L0 = 7 epsilon = 2.3401672840118404
L0 = 8 epsilon = 3.6779388308525074
L0 = 9 epsilon = 5.562909078598022
L0 = 10 epsilon = 8.330916881561276


In [35]:
test_um = perm_abc_smc_um(key = key, model = model, n_particles= n_particles, y_obs= y_obs, kernel = kernel, L_0 = 7, epsilon = epsilon, verbose = 3, update_weights_distance= False)

a) Simulation of the first particles:
b) Computing the first distances with L = 7: Performing full optimal assignment (no prior indices).
min = 0.45 max = 5.1 mean = 1.3
d) Setting the first weights: ESS = 866
Iteration 0: L = 7 Epsilon = 1.73603187084198, ESS = 866 Acc. rate = 100% Numb. unique particles = 1000

a) Update L: new L = 8 old L = 7
b) Compute optimal distances: 866 unique particles Performing full optimal assignment (no prior indices).
c) Update weights: Old ESS = 866 New ESS = 539 (37.76% of the particles killed)
f) Resampling: No resampling
d) Move particles:
1. Forward kernel: 
Silo 0: 76.07% matched! Thetas_match: min = 7.22, max = 15.7 mean = 11.2
Silo 1: 82.19% matched! Thetas_match: min = -12.2, max = -4.12 mean = -7.83
Silo 2: 83.49% matched! Thetas_match: min = -8.6, max = -1.76 mean = -5.14
Silo 3: 86.64% matched! Thetas_match: min = -0.611, max = 6.08 mean = 2.48
Silo 4: 71.24% matched! Thetas_match: min = -15.7, max = -6.87 mean = -11.1
Silo 5: 85.53% matched!

In [ ]:
print("Number of simulations:\npermSMC = {:.2e} permOS (no duplicate) = {:.2e} permOS (duplicate) = {:.2e} permUM = {:.2e} \n".format(np.sum(test_permsmc["N_sim"]), np.sum(test_os["N_sim"]), np.sum(test_os_duplicate["N_sim"]), np.sum(test_um["N_sim"])))
print("Unique particle rate:\npermSMC = {:.2%} permOS (no duplicate) = {:.2%} permOS (duplicate) = {:.2%} permUM = {:.2%}\n".format(test_permsmc["unique_part"][-1], test_os["unique_part"][-1], test_os_duplicate["unique_part"][-1],test_um["unique_part"][-1]))
print("Number of simulations per unique particles:\npermSMC = {:.2e} permOS (no duplicate) = {:.2e} permOS (duplicate) = {:.2e} permUM = {:.2e}\n".format(np.sum(test_permsmc["N_sim"])/test_permsmc["unique_part"][-1]/n_particles, np.sum(test_os["N_sim"])/test_os["unique_part"][-1]/n_particles, np.sum(test_os_duplicate["N_sim"])/test_os_duplicate["unique_part"][-1]/n_particles,  np.sum(test_um["N_sim"])/test_um["unique_part"][-1]/n_particles))
print("Time:\npermSMC = {:.2e} permOS (no duplicate) = {:.2e} permOS (duplicate) = {:.2e} permUM = {:.2e}\n".format(np.sum(test_permsmc["Time"]), np.sum(test_os["Time"]), np.sum(test_os_duplicate["Time"]), np.sum(test_um["Time"])))

Number of simulations:
permSMC = 6.52e+05 permOS (no duplicate) = 2.32e+05 permOS (duplicate) = 2.60e+05 permUM = 3.54e+04 

Unique particle rate:
permSMC = 77.00% permOS (no duplicate) = 24.20% permOS (duplicate) = 88.60% permUM = 19.70%

Number of simulations per unique particles:
permSMC = 8.47e+02 permOS (no duplicate) = 9.60e+02 permOS (duplicate) = 2.93e+02 permUM = 1.80e+02

Time:
permSMC = 2.08e+02 permOS (no duplicate) = 6.29e+01 permOS (duplicate) = 1.67e+01 permUM = 1.02e+01



<function print>